## Homework 9: Text Classification with Fine-Tuned BERT

In this final homework, we’ll explore **fine-tuning a pre-trained Transformer model (BERT)** for text classification using the **IMDB Movie Review** dataset. You’ll begin with a working baseline notebook and then conduct a series of controlled experiments to understand how data size, context length, and model architecture affect performance.

You’ll complete three problems:

* **Problem 1:** Evaluate how **sequence length** and **learning rate** jointly influence validation loss and generalization.
* **Problem 2:** Measure how **training data size** affects both model performance and total training time.
* **Problem 3:** Compare **two additional models** from the BERT family to analyze the trade-offs between model size and accuracy on this dataset.

In each problem, you’ll report your key metrics, summarize what you observed, and reflect on what you learned.

> **Note:** This homework was developed and tested on **Google Colab**, due to version conflicts when running locally. It is **strongly recommended** that you complete your work on Colab as well.

There are 6 problems, each worth 14 points, and you get one point free if you complete the entire homework.


In [1]:
# Install once per new Colab runtime
#%pip -q install -U keras keras-hub tensorflow tensorflow-text datasets evaluate
%pip install keras keras-hub tensorflow-text datasets evaluate
%pip install tensorflow-text==2.19.*

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [2]:

import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time
import random
import numpy as np
import keras
import keras_hub as kh
import evaluate
from datasets import load_dataset, Dataset, Features, Value, ClassLabel

from keras import mixed_precision                    # generally faster
mixed_precision.set_global_policy("mixed_float16")

### Here is where you can set global hyperparameters for this homework

In [3]:
# ---------------- Config ----------------
SEED        = 42
MAX_LEN     = 512
EPOCHS      = 3
BATCH       = 32
EVAL_BATCH  = 64
SUBSET_FRAC = 0.25   # <-- 0.25 to train and test on 25% of whole dataset during development;  set to 1.0 for full dataset

keras.utils.set_random_seed(SEED)

### Load and Preprocess the IMDB Movie Review Dataset

In [4]:
# ---- Load IMDb (raw), join train+test ----
imdb   = load_dataset("imdb")
texts  = list(imdb["train"]["text"]) + list(imdb["test"]["text"])
labels = np.array(list(imdb["train"]["label"]) + list(imdb["test"]["label"]), dtype="int32")

# ---- Build DS with explicit features (label=ClassLabel) ----
features = Features({"text": Value("string"),
                     "label": ClassLabel(num_classes=2, names=["NEG","POS"])})
all_ds = Dataset.from_dict({"text": texts, "label": labels.tolist()}, features=features)

# ---- Optional: take a stratified subset of the FULL dataset ----
if 0.0 < SUBSET_FRAC < 1.0:
    sub = all_ds.train_test_split(train_size=SUBSET_FRAC, seed=SEED, stratify_by_column="label")
    ds_pool = sub["train"]
else:
    ds_pool = all_ds

# ---- Stratified 80/10/10 split on the (possibly smaller) pool ----
# First: 80/20 train+val pool / test
splits = ds_pool.train_test_split(test_size=0.20, seed=SEED, stratify_by_column="label")
train_val_pool, test_ds = splits["train"], splits["test"]
# Then: carve 10% of full (i.e., 0.125 of the 80% pool) as validation
splits2 = train_val_pool.train_test_split(test_size=0.125, seed=SEED, stratify_by_column="label")
train_ds, val_ds = splits2["train"], splits2["test"]

# ---- Numpy arrays for Keras fit/predict ----
X_tr = np.array(train_ds["text"], dtype=object); y_tr = np.array(train_ds["label"], dtype="int32")
X_va = np.array(val_ds["text"],   dtype=object); y_va = np.array(val_ds["label"],   dtype="int32")
X_te = np.array(test_ds["text"],  dtype=object); y_te = np.array(test_ds["label"],  dtype="int32")

# ---- Quick summary ----
def _counts(ds):
    arr = np.array(ds["label"], dtype=int)
    return len(arr), np.bincount(arr, minlength=2).tolist()
print(f"Pool after SUBSET_FRAC={SUBSET_FRAC}: {len(ds_pool)} (of {len(all_ds)})")
print("Train:", _counts(train_ds), " Val:", _counts(val_ds), " Test:", _counts(test_ds))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Pool after SUBSET_FRAC=0.25: 12500 (of 50000)
Train: (8750, [4375, 4375])  Val: (1250, [625, 625])  Test: (2500, [1250, 1250])


### Build and train a baseline Distil-Bert Text Classifier

In [5]:
# ---- Keras Hub preprocessor + classifier ----
preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased", sequence_length=MAX_LEN
)
model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

start = time.time()

# ---- Train with early stopping (restore best val weights) ----
cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=1,
)

# ---- Evaluate (accuracy + F1 via `evaluate`) ----
logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
y_pred = logits.argmax(axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")
acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

# Tiny confusion matrix helper (no sklearn needed)
def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

print(f"\nValidation acc (best epoch): {history.history['val_acc'][np.argmin(history.history['val_loss'])]:.3f}")
print(f"\nTest accuracy: {acc:.3f}   Test F1: {f1:.3f}")
print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))

end = time.time() - start
print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(end)))

Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 198s 549ms/step - acc: 0.8305 - loss: 0.3693 - val_acc: 0.9096 - val_loss: 0.2315
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 131s 477ms/step - acc: 0.9274 - loss: 0.1932 - val_acc: 0.9136 - val_loss: 0.2230
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 130s 473ms/step - acc: 0.9487 - loss: 0.1379 - val_acc: 0.9000 - val_loss: 0.2593



Validation acc (best epoch): 0.914

Test accuracy: 0.914   Test F1: 0.914

Confusion matrix:
 [[1129  121]
 [  95 1155]]

Elapsed time: 00:08:05


# Problem 1 — Mini sweep: context length × learning rate (6 runs)

In this problem we'll see how much **context length** (`MAX_LEN`) helps, and how sensitive fine-tuning is to **learning rate**—without running a huge grid.

## Setup (keep these fixed)

* `SUBSET_FRAC = 0.25`               # use only this percentage of the whole dataset
* `EPOCHS = 3`
* `BATCH = 32` (but see note for 256 below)
* **EarlyStopping** with `restore_best_weights=True`
* Same random `SEED` for all runs
* Same data split for all runs (don’t reshuffle between runs)

### Run these 6 configurations

**For each** `MAX_LEN ∈ {128, 256, 512}`, try **two** learning rates:

* **MAX_LEN = 128**

  * `(LR = 2e-5, BATCH = 32)` – healthy default for shorter contexts.
  * `(LR = 1e-5, BATCH = 32)` – conservative LR; often a touch stabler.

* **MAX_LEN = 256**

  * `(LR = 1e-5, BATCH = 16)` – longer context → lower batch.
  * `(LR = 7.5e-6, BATCH = 16)` – even steadier if loss is noisy.

* **MAX_LEN = 512**  *(heavier quadratic attention cost)*

  * `(LR = 7.5e-6, BATCH = 8)` – safe starting point.
  * `(LR = 5e-6, BATCH = 8)` – extra caution for stability.

**If you hit an Out Of Memory error:**

* At **256** with `BATCH = 16`, drop to `BATCH = 8`.
* At **512** with `BATCH = 8`, drop to `BATCH = 4`.


Then answer the graded questions.


In [ ]:
# Your code here; add as many cells as you need
## set up
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def p1_run_experiment(run_name, max_len, lr, batch):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)

    start = time.time()

    preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
        "distil_bert_base_en_uncased",
        sequence_length=max_len
    )

    model = kh.models.DistilBertTextClassifier.from_preset(
        "distil_bert_base_en_uncased",
        num_classes=2,
        preprocessor=preproc
    )

    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    cb = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=batch,
        callbacks=cb,
        verbose=1,
    )

    logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
    y_pred = logits.argmax(axis=-1)

    acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
    f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

    best_idx      = int(np.argmin(history.history["val_loss"]))
    best_epoch    = best_idx + 1
    best_val_loss = float(history.history["val_loss"][best_idx])
    best_val_acc  = float(history.history["val_acc"][best_idx])

    elapsed = time.time() - start

    print(f"\n{run_name}")
    print(f"Best epoch: {best_epoch}")
    print(f"Validation acc (best epoch): {best_val_acc:.3f}")
    print(f"Validation loss (best epoch): {best_val_loss:.3f}")
    print(f"Test accuracy: {acc:.3f}")
    print(f"Test F1: {f1:.3f}")
    print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))
    print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(elapsed)))

    return {
        "run_name": run_name,
        "MAX_LEN": max_len,
        "LR": lr,
        "BATCH": batch,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_acc": acc,
        "test_f1": f1,
        "elapsed_sec": elapsed,
    }

In [ ]:
## prob 1 run 1
p1_run1 = p1_run_experiment(
    run_name="Run 1: MAX_LEN=128, LR=2e-5, BATCH=32",
    max_len=128,
    lr=2e-5,
    batch=32
)

Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 174s 501ms/step - acc: 0.8043 - loss: 0.4147 - val_acc: 0.8528 - val_loss: 0.3411
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 119s 431ms/step - acc: 0.8977 - loss: 0.2510 - val_acc: 0.8496 - val_loss: 0.3772
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 119s 432ms/step - acc: 0.9360 - loss: 0.1723 - val_acc: 0.8600 - val_loss: 0.3866

Run 1: MAX_LEN=128, LR=2e-5, BATCH=32
Best epoch: 1
Validation acc (best epoch): 0.853
Validation loss (best epoch): 0.341
Test accuracy: 0.849
Test F1: 0.855

Confusion matrix:
 [[1004  246]
 [ 132 1118]]

Elapsed time: 00:07:30


In [ ]:
## prob 1 run 2
p2_run2 = p1_run_experiment(
    run_name="Run 2: MAX_LEN=128, LR=1e-5, BATCH=32",
    max_len=128,
    lr=1e-5,
    batch=32
)

100%|██████████| 794/794 [00:00<00:00, 2.03MB/s]


100%|██████████| 462/462 [00:00<00:00, 718kB/s]


100%|██████████| 253M/253M [00:21<00:00, 12.1MB/s]


Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 172s 500ms/step - acc: 0.7797 - loss: 0.4543 - val_acc: 0.8584 - val_loss: 0.3384
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 119s 434ms/step - acc: 0.8825 - loss: 0.2863 - val_acc: 0.8640 - val_loss: 0.3407
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 119s 433ms/step - acc: 0.9185 - loss: 0.2182 - val_acc: 0.8648 - val_loss: 0.3633

Run 2: MAX_LEN=128, LR=1e-5, BATCH=32
Best epoch: 1
Validation acc (best epoch): 0.858
Validation loss (best epoch): 0.338
Test accuracy: 0.845
Test F1: 0.845

Confusion matrix:
 [[1053  197]
 [ 191 1059]]

Elapsed time: 00:08:17


In [ ]:
## prob 1 run 3
p3_run3 = p1_run_experiment(
    run_name="Run 3: MAX_LEN=256, LR=1e-5, BATCH=16",
    max_len=256,
    lr=1e-5,
    batch=16
)

Epoch 1/3
547/547 ━━━━━━━━━━━━━━━━━━━━ 283s 451ms/step - acc: 0.8342 - loss: 0.3623 - val_acc: 0.8968 - val_loss: 0.2434
Epoch 2/3
547/547 ━━━━━━━━━━━━━━━━━━━━ 222s 406ms/step - acc: 0.9181 - loss: 0.2108 - val_acc: 0.9064 - val_loss: 0.2436
Epoch 3/3
547/547 ━━━━━━━━━━━━━━━━━━━━ 222s 406ms/step - acc: 0.9454 - loss: 0.1447 - val_acc: 0.9072 - val_loss: 0.2671

Run 3: MAX_LEN=256, LR=1e-5, BATCH=16
Best epoch: 1
Validation acc (best epoch): 0.897
Validation loss (best epoch): 0.243
Test accuracy: 0.888
Test F1: 0.888

Confusion matrix:
 [[1108  142]
 [ 139 1111]]

Elapsed time: 00:12:55


In [ ]:
## prob 1 run 4
p1_run4 = p1_run_experiment(
    run_name="Run 4: MAX_LEN=256, LR=7.5e-6, BATCH=8",
    max_len=256,
    lr=7.5e-6,
    batch=8
)

Epoch 1/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 291s 233ms/step - acc: 0.8401 - loss: 0.3487 - val_acc: 0.9040 - val_loss: 0.2383
Epoch 2/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 241s 220ms/step - acc: 0.9210 - loss: 0.2058 - val_acc: 0.9056 - val_loss: 0.2425
Epoch 3/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 230s 210ms/step - acc: 0.9493 - loss: 0.1402 - val_acc: 0.9008 - val_loss: 0.2825

Run 4: MAX_LEN=256, LR=7.5e-6, BATCH=8
Best epoch: 1
Validation acc (best epoch): 0.904
Validation loss (best epoch): 0.238
Test accuracy: 0.889
Test F1: 0.890

Confusion matrix:
 [[1094  156]
 [ 122 1128]]

Elapsed time: 00:13:29


In [ ]:
## prob 1 run 5
p1_run5 = p1_run_experiment(
    run_name="Run 5: MAX_LEN=512, LR=7.5e-6, BATCH=4",
    max_len=512,
    lr=7.5e-6,
    batch=4
)

Epoch 1/3
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 543s 231ms/step - acc: 0.8661 - loss: 0.3059 - val_acc: 0.9016 - val_loss: 0.2560
Epoch 2/3
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 478s 218ms/step - acc: 0.9409 - loss: 0.1644 - val_acc: 0.9216 - val_loss: 0.2381
Epoch 3/3
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 478s 218ms/step - acc: 0.9663 - loss: 0.0981 - val_acc: 0.9216 - val_loss: 0.2651

Run 5: MAX_LEN=512, LR=7.5e-6, BATCH=4
Best epoch: 2
Validation acc (best epoch): 0.922
Validation loss (best epoch): 0.238
Test accuracy: 0.916
Test F1: 0.917

Confusion matrix:
 [[1135  115]
 [  94 1156]]

Elapsed time: 00:26:17


In [ ]:
## prob 1 run 6
p1_run6 = p1_run_experiment(
    run_name="Run 6: MAX_LEN=512, LR=5e-6, BATCH=4",
    max_len=512,
    lr=5e-6,
    batch=4
)

Epoch 1/3
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 537s 229ms/step - acc: 0.8616 - loss: 0.3190 - val_acc: 0.9080 - val_loss: 0.2318
Epoch 2/3
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 479s 219ms/step - acc: 0.9342 - loss: 0.1799 - val_acc: 0.9160 - val_loss: 0.2222
Epoch 3/3
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 478s 218ms/step - acc: 0.9584 - loss: 0.1197 - val_acc: 0.9128 - val_loss: 0.2490

Run 6: MAX_LEN=512, LR=5e-6, BATCH=4
Best epoch: 2
Validation acc (best epoch): 0.916
Validation loss (best epoch): 0.222
Test accuracy: 0.919
Test F1: 0.920

Confusion matrix:
 [[1138  112]
 [  90 1160]]

Elapsed time: 00:26:03


In [ ]:
## compare
import pandas as pd
p1_results_df = pd.DataFrame([p1_run1, p2_run2, p3_run3, p1_run4, p1_run5, p1_run6])
p1_results_df = p1_results_df.sort_values(
    by=["test_f1", "test_acc", "best_val_acc"],
    ascending=False
).reset_index(drop=True)

display(p1_results_df)

,run_name,MAX_LEN,LR,BATCH,best_epoch,best_val_loss,best_val_acc,test_acc,test_f1,elapsed_sec
0,"Run 6: MAX_LEN=512, LR=5e-6, BATCH=4",512,0.000005,4,2,0.222156,0.9160,0.9192,0.919905,1563.989720
1,"Run 5: MAX_LEN=512, LR=7.5e-6, BATCH=4",512,0.000008,4,2,0.238130,0.9216,0.9164,0.917096,1577.040601
2,"Run 4: MAX_LEN=256, LR=7.5e-6, BATCH=8",256,0.000008,8,1,0.238291,0.9040,0.8888,0.890292,809.380386
3,"Run 3: MAX_LEN=256, LR=1e-5, BATCH=16",256,0.000010,16,1,0.243371,0.8968,0.8876,0.887735,775.948781
4,"Run 1: MAX_LEN=128, LR=2e-5, BATCH=32",128,0.000020,32,1,0.341071,0.8528,0.8488,0.855394,450.619173
5,"Run 2: MAX_LEN=128, LR=1e-5, BATCH=32",128,0.000010,32,1,0.338364,0.8584,0.8448,0.845172,497.579805


### Graded Questions

In [ ]:
# Set a1a to the validation accuracy at min validation loss for your best configuration found in this problem

a1a = 0.9160             # Replace 0.0 with your answer

In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a1a = {a1a:.4f}')

a1a = 0.9160


#### Question a1b:

* Does **more context** (128 → 256 → 512) consistently help?
* How much effect did the learning rate have on the validation accuracy?


#### Your Answer Here:
Yes more context consistently helped the models validation accuracy.
Generally speaking, from the results table, a smaller learning rate was beneficial for validation accuracy while a larger learning rate hurt the validation accuracy.

## Problem 2 — How much data is enough?

In this problem, you’ll investigate how model performance scales with dataset size.

**Setup.**
Use the best `MAX_LEN` and `LR` values you found in **Problem 1**.

**What to do:**

1. For each value of `SUBSET_FRAC ∈ {0.25, 0.50, 0.75, 1.00}`, train your model once and observe the displayed performance metrics.
2. Answer the discussion question below.




In [ ]:
# Your code here; add as many cells as you need
## set up
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def p2_run_experiment(run_name, subset_frac, max_len, lr, batch):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)

    start = time.time()

    # ---- create subset for this run ----
    if 0.0 < subset_frac < 1.0:
        sub = all_ds.train_test_split(
            train_size=subset_frac,
            seed=SEED,
            stratify_by_column="label"
        )
        ds_pool = sub["train"]
    else:
        ds_pool = all_ds

    # ---- fixed split logic ----
    splits = ds_pool.train_test_split(
        test_size=0.20,
        seed=SEED,
        stratify_by_column="label"
    )
    train_val_pool, test_ds = splits["train"], splits["test"]

    splits2 = train_val_pool.train_test_split(
        test_size=0.125,   # gives 10% of full pool as validation
        seed=SEED,
        stratify_by_column="label"
    )
    train_ds, val_ds = splits2["train"], splits2["test"]

    # ---- numpy arrays for this run ----
    X_tr = np.array(train_ds["text"], dtype=object)
    y_tr = np.array(train_ds["label"], dtype="int32")
    X_va = np.array(val_ds["text"], dtype=object)
    y_va = np.array(val_ds["label"], dtype="int32")
    X_te = np.array(test_ds["text"], dtype=object)
    y_te = np.array(test_ds["label"], dtype="int32")

    preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
        "distil_bert_base_en_uncased",
        sequence_length=max_len
    )

    model = kh.models.DistilBertTextClassifier.from_preset(
        "distil_bert_base_en_uncased",
        num_classes=2,
        preprocessor=preproc
    )

    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    cb = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=batch,
        callbacks=cb,
        verbose=1,
    )

    logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
    y_pred = logits.argmax(axis=-1)

    acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
    f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

    best_idx      = int(np.argmin(history.history["val_loss"]))
    best_epoch    = best_idx + 1
    best_val_loss = float(history.history["val_loss"][best_idx])
    best_val_acc  = float(history.history["val_acc"][best_idx])

    elapsed = time.time() - start

    print(f"\n{run_name}")
    print(f"SUBSET_FRAC: {subset_frac}")
    print(f"Pool size: {len(ds_pool)}")
    print(f"Train size: {len(train_ds)}   Val size: {len(val_ds)}   Test size: {len(test_ds)}")
    print(f"Best epoch: {best_epoch}")
    print(f"Validation acc (best epoch): {best_val_acc:.3f}")
    print(f"Validation loss (best epoch): {best_val_loss:.3f}")
    print(f"Test accuracy: {acc:.3f}")
    print(f"Test F1: {f1:.3f}")
    print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))
    print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(elapsed)))

    return {
        "run_name": run_name,
        "SUBSET_FRAC": subset_frac,
        "MAX_LEN": max_len,
        "LR": lr,
        "BATCH": batch,
        "pool_size": len(ds_pool),
        "train_size": len(train_ds),
        "val_size": len(val_ds),
        "test_size": len(test_ds),
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_acc": acc,
        "test_f1": f1,
        "elapsed_sec": elapsed,
    }

In [ ]:
from datasets import Dataset, Features, Value, ClassLabel

imdb   = load_dataset("imdb")
texts  = list(imdb["train"]["text"]) + list(imdb["test"]["text"])
labels = np.array(list(imdb["train"]["label"]) + list(imdb["test"]["label"]), dtype="int32")

features = Features({
    "text": Value("string"),
    "label": ClassLabel(num_classes=2, names=["NEG","POS"])
})

all_ds = Dataset.from_dict(
    {"text": texts, "label": labels.tolist()},
    features=features
)

In [ ]:
# Replace these with the best settings from Problem 1
BEST_MAX_LEN = 512
BEST_LR = 5e-5
BEST_BATCH = 32

p2_results = []

for frac in [0.25, 0.50, 0.75, 1.00]:
    res = p2_run_experiment(
        run_name=f"P2 | SUBSET_FRAC={frac}",
        subset_frac=frac,
        max_len=BEST_MAX_LEN,
        lr=BEST_LR,
        batch=BEST_BATCH
    )
    p2_results.append(res)

Epoch 1/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 574s 2s/step - acc: 0.8627 - loss: 0.3110 - val_acc: 0.9048 - val_loss: 0.2339
Epoch 2/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 496s 2s/step - acc: 0.9424 - loss: 0.1544 - val_acc: 0.9048 - val_loss: 0.2690
Epoch 3/3
274/274 ━━━━━━━━━━━━━━━━━━━━ 498s 2s/step - acc: 0.9691 - loss: 0.0877 - val_acc: 0.8888 - val_loss: 0.3336

P2 | SUBSET_FRAC=0.25
SUBSET_FRAC: 0.25
Pool size: 12500
Train size: 8750   Val size: 1250   Test size: 2500
Best epoch: 1
Validation acc (best epoch): 0.905
Validation loss (best epoch): 0.234
Test accuracy: 0.909
Test F1: 0.908

Confusion matrix:
 [[1144  106]
 [ 122 1128]]

Elapsed time: 00:27:31
Epoch 1/3
547/547 ━━━━━━━━━━━━━━━━━━━━ 1063s 2s/step - acc: 0.8854 - loss: 0.2740 - val_acc: 0.9136 - val_loss: 0.2195
Epoch 2/3
547/547 ━━━━━━━━━━━━━━━━━━━━ 990s 2s/step - acc: 0.9427 - loss: 0.1536 - val_acc: 0.9040 - val_loss: 0.3006
Epoch 3/3
547/547 ━━━━━━━━━━━━━━━━━━━━ 993s 2s/step - acc: 0.9667 - loss: 0.0957 - val_acc: 0.9208 - val_lo

### Graded Questions

In [ ]:
# Set a2a to the validation accuracy at min validation loss for your best configuration found in this problem
# (Yes, it is probably at 1.0!)

a2a = 0.9300             # Replace 0.0 with your answer

In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a2a = {a2a:.4f}')

a2a = 0.9300


#### Question a2b:

Summarize what you observed as dataset size increased. Given that validation metrics are typically reliable to only about two decimal places, do the performance gains justify using the entire dataset? What trade-offs between accuracy and computation time did you notice?

#### Your Answer Here:

# Problem 3 — Model swap: speed vs. accuracy (why: capacity matters)

In this problem we will compare encoder-only backbones of different sizes.

**Setup.** Keep the best `MAX_LEN`, `LR`, and `SUBSET_FRAC` from Problems 1–2. Only change the model/preset:

* **DistilBERT** (current baseline)
* **BERT-base** (larger/usually stronger)

**How to switch (two lines each).**

* DistilBERT:

  ```python
  preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset("distil_bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.DistilBertTextClassifier.from_preset("distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```

* BERT-base:

  ```python
  preproc = kh.models.BertTextClassifierPreprocessor.from_preset("bert_base_en_uncased", sequence_length=MAX_LEN)
  model  = kh.models.BertTextClassifier.from_preset("bert_base_en_uncased", num_classes=2, preprocessor=preproc)
  ```

**What to do.**

1. Train/evaluate each model once with identical settings.
2. Observe the performance metrics for each.
3. Answer the graded questions.



In [6]:
# Your code here; add as many cells as you wish
## set up
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def p3_run_experiment(run_name, subset_frac, max_len, lr, batch, backbone):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)

    start = time.time()

    # ---- create subset for this run ----
    if 0.0 < subset_frac < 1.0:
        sub = all_ds.train_test_split(
            train_size=subset_frac,
            seed=SEED,
            stratify_by_column="label"
        )
        ds_pool = sub["train"]
    else:
        ds_pool = all_ds

    # ---- fixed 80/10/10 split ----
    splits = ds_pool.train_test_split(
        test_size=0.20,
        seed=SEED,
        stratify_by_column="label"
    )
    train_val_pool, test_ds = splits["train"], splits["test"]

    splits2 = train_val_pool.train_test_split(
        test_size=0.125,
        seed=SEED,
        stratify_by_column="label"
    )
    train_ds, val_ds = splits2["train"], splits2["test"]

    # ---- numpy arrays ----
    X_tr = np.array(train_ds["text"], dtype=object)
    y_tr = np.array(train_ds["label"], dtype="int32")
    X_va = np.array(val_ds["text"], dtype=object)
    y_va = np.array(val_ds["label"], dtype="int32")
    X_te = np.array(test_ds["text"], dtype=object)
    y_te = np.array(test_ds["label"], dtype="int32")

    # ---- choose model ----
    if backbone == "distilbert":
        preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
            "distil_bert_base_en_uncased",
            sequence_length=max_len
        )
        model = kh.models.DistilBertTextClassifier.from_preset(
            "distil_bert_base_en_uncased",
            num_classes=2,
            preprocessor=preproc
        )

    elif backbone == "bert":
        preproc = kh.models.BertTextClassifierPreprocessor.from_preset(
            "bert_base_en_uncased",
            sequence_length=max_len
        )
        model = kh.models.BertTextClassifier.from_preset(
            "bert_base_en_uncased",
            num_classes=2,
            preprocessor=preproc
        )

    else:
        raise ValueError("backbone must be 'distilbert' or 'bert'")

    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )

    cb = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=batch,
        callbacks=cb,
        verbose=1,
    )

    logits = model.predict(X_te, batch_size=EVAL_BATCH, verbose=0)
    y_pred = logits.argmax(axis=-1)

    acc = acc_metric.compute(predictions=y_pred, references=y_te)["accuracy"]
    f1  = f1_metric.compute(predictions=y_pred, references=y_te)["f1"]

    best_idx      = int(np.argmin(history.history["val_loss"]))
    best_epoch    = best_idx + 1
    best_val_loss = float(history.history["val_loss"][best_idx])
    best_val_acc  = float(history.history["val_acc"][best_idx])

    elapsed = time.time() - start

    print(f"\n{run_name}")
    print(f"Backbone: {backbone}")
    print(f"SUBSET_FRAC: {subset_frac}")
    print(f"MAX_LEN: {max_len}   LR: {lr}   BATCH: {batch}")
    print(f"Pool size: {len(ds_pool)}")
    print(f"Train size: {len(train_ds)}   Val size: {len(val_ds)}   Test size: {len(test_ds)}")
    print(f"Best epoch: {best_epoch}")
    print(f"Validation acc (best epoch): {best_val_acc:.3f}")
    print(f"Validation loss (best epoch): {best_val_loss:.3f}")
    print(f"Test accuracy: {acc:.3f}")
    print(f"Test F1: {f1:.3f}")
    print("\nConfusion matrix:\n", confusion_matrix_np(y_te, y_pred))
    print("\nElapsed time:", time.strftime("%H:%M:%S", time.gmtime(elapsed)))

    return {
        "run_name": run_name,
        "backbone": backbone,
        "SUBSET_FRAC": subset_frac,
        "MAX_LEN": max_len,
        "LR": lr,
        "BATCH": batch,
        "pool_size": len(ds_pool),
        "train_size": len(train_ds),
        "val_size": len(val_ds),
        "test_size": len(test_ds),
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_acc": acc,
        "test_f1": f1,
        "elapsed_sec": elapsed,
    }

In [ ]:
# Replace these with your actual best settings from Problems 1 and 2
BEST_SUBSET_FRAC = 1.00
BEST_MAX_LEN = 512
BEST_LR = 0.000005
BEST_BATCH = 4

p3_results = []

res_distil = p3_run_experiment(
    run_name="P3 | DistilBERT",
    subset_frac=BEST_SUBSET_FRAC,
    max_len=BEST_MAX_LEN,
    lr=BEST_LR,
    batch=BEST_BATCH,
    backbone="distilbert"
)
p3_results.append(res_distil)

res_bert = p3_run_experiment(
    run_name="P3 | BERT-base",
    subset_frac=BEST_SUBSET_FRAC,
    max_len=BEST_MAX_LEN,
    lr=BEST_LR,
    batch=BEST_BATCH,
    backbone="bert"
)
p3_results.append(res_bert)

Epoch 1/3
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 1977s 222ms/step - acc: 0.8973 - loss: 0.2485 - val_acc: 0.9330 - val_loss: 0.1796
Epoch 2/3
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 1945s 222ms/step - acc: 0.9425 - loss: 0.1572 - val_acc: 0.9320 - val_loss: 0.1910
Epoch 3/3
4602/8750 ━━━━━━━━━━━━━━━━━━━━ 14:35 211ms/step - acc: 0.9572 - loss: 0.1173

In [ ]:
p3_results_df = pd.DataFrame(p3_results)
display(p3_results_df.sort_values("test_f1", ascending=False))

### Graded Questions

In [ ]:
# Set a1a to the validation accuracy at min validation loss for your best model found in this problem

a3a = 0.0             # Replace 0.0 with your answer

In [ ]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a3a = {a3a:.4f}')

a3a = 0.0000


#### Question a3b:

**Answer briefly.**

* Which model gives the best **accuracy/F1**?
* Which is **fastest** per epoch?
* Given limited development time or compute resources, which model is the best **overall choice** and why?

#### Your Answer Here: